# GeoMapBench — validated multimodal RAG evaluation

This notebook runs two new conditions: `multimodal_rag` and `agentic_multimodal_rag`. Both use BGE text retrieval, the existing CLIP image index, benchmark-image-to-corpus-image retrieval, rank fusion, optional retrieved reference images, and a predeclared 14-task coverage gate. BM25 is absent.

The paid answer evaluation cannot start until a runtime validation proves that both the 180,344-record text index and the 1,794-image CLIP index are loaded, a benchmark image is encoded, both searches return results, fusion succeeds, and retrieved corpus image files are accessible. The old 1,662-asset benchmark preflight is not rerun; its already-passed report is trusted for the portable benchmark hash.

Old text-only RAG results are preserved as an ablation and are never imported into this new output. Re-running the execution cell resumes only the v2.2 multimodal conditions.

In [ ]:
# EDIT ONLY THIS CELL
GITHUB_REPO = "https://github.com/asalmeskin/GeoMapBench.git"
GIT_REF = "main"

BENCHMARK_ROOT = "/content/drive/MyDrive/GeoMapBench_Data/geomapbench_100"
CORPUS_ROOT = "/content/drive/MyDrive/GeoMapRAG_Corpus"
RESULTS_ROOT = "/content/drive/MyDrive/geomapbench_results_final"
CACHE_ROOT = "/content/drive/MyDrive/geomapbench_runtime_cache"

ANSWER_MODEL = "anthropic/claude-sonnet-5"
AGENT_MODEL = "google/gemini-3.5-flash-lite"
TARGET_PER_LEAF = 1  # smoke: 1; then increase in-place to 6, 50, or 100
MAX_COST_USD_PER_CONDITION = 30.0
REQUEST_DELAY_SECONDS = 1.0
PROGRESS_EVERY = 5

FINAL_OUTPUT_NAME = "rag_suite_multimodal_claude_v220"
TRUSTED_BENCHMARK_REPORT = f"{CACHE_ROOT}/preflight_final/benchmark_preflight.json"
BASE_RESULTS = f"{RESULTS_ROOT}/model_suite_final/anthropic_claude-sonnet-5/responses.jsonl"
INSTALL_RAG = True

In [ ]:
def run_live(command, *, cwd=None, env_extra=None, display_command=None):
    import os, subprocess
    environment = os.environ.copy()
    environment.update(env_extra or {})
    environment["PYTHONUNBUFFERED"] = "1"
    shown = display_command or list(map(str, command))
    print("Running:", " ".join(map(str, shown)), flush=True)
    process = subprocess.Popen(
        list(map(str, command)), cwd=str(cwd) if cwd else None, env=environment,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
    if return_code:
        raise RuntimeError(f"Command failed with exit code {return_code}")


def clone_repository(destination, repository, ref):
    import shutil
    from pathlib import Path

    destination = Path(destination)
    staging = destination.with_name(destination.name + "_staging")
    if staging.exists():
        shutil.rmtree(staging)
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None
    clone_url = repository
    shown_url = repository
    if token and repository.startswith("https://github.com/"):
        clone_url = repository.replace("https://", f"https://x-access-token:{token}@", 1)
        shown_url = repository + " (using Colab secret GITHUB_TOKEN)"
    command = ["git", "clone", "--depth", "1", "--branch", ref, clone_url, str(staging)]
    shown = ["git", "clone", "--depth", "1", "--branch", ref, shown_url, str(staging)]
    try:
        run_live(command, env_extra={"GIT_TERMINAL_PROMPT": "0"}, display_command=shown)
    except Exception as error:
        raise RuntimeError(
            "Git clone failed. Confirm GIT_REF exists on GitHub. If the repository is private, "
            "add a Colab secret named GITHUB_TOKEN with read access and rerun this cell."
        ) from error
    if destination.exists():
        shutil.rmtree(destination)
    staging.rename(destination)

from google.colab import drive
drive.mount("/content/drive")

import os, subprocess, sys
from pathlib import Path

repo = Path("/content/GeoMapBench")
clone_repository(repo, GITHUB_REPO, GIT_REF)
INSTALL_TARGET = f"{repo}[rag-index]" if INSTALL_RAG else str(repo)
run_live([sys.executable, "-m", "pip", "install", "-q", "-e", INSTALL_TARGET])

assert Path(BENCHMARK_ROOT).is_dir(), f"Benchmark not found: {BENCHMARK_ROOT}"
Path(RESULTS_ROOT).mkdir(parents=True, exist_ok=True)
Path(CACHE_ROOT).mkdir(parents=True, exist_ok=True)
os.environ["GEOMAPBENCH_IMAGE_CACHE"] = str(Path(CACHE_ROOT) / "converted_images_final")
os.environ["HF_HOME"] = str(Path(CACHE_ROOT) / "huggingface")
os.environ["SENTENCE_TRANSFORMERS_HOME"] = str(Path(CACHE_ROOT) / "huggingface" / "sentence_transformers")
os.environ["PYTHONUNBUFFERED"] = "1"
version = subprocess.check_output(
    [sys.executable, "-c", "import geomapbench_eval; print(geomapbench_eval.__version__)"], text=True,
).strip()
assert version == "2.2.0", f"Expected GeoMapBench 2.2.0, installed {version}"
print("Installed final release:", version)

assert Path(CORPUS_ROOT).is_dir(), f"RAG corpus not found: {CORPUS_ROOT}"
for relative in (
    "indexes/text.faiss", "indexes/text_metadata.jsonl", "indexes/text_manifest.json",
    "indexes/image.faiss", "indexes/image_metadata.jsonl", "indexes/image_manifest.json",
):
    assert (Path(CORPUS_ROOT) / relative).is_file(), f"Missing multimodal artifact: {relative}"
assert Path(TRUSTED_BENCHMARK_REPORT).is_file(), (
    "The already-passed benchmark report is missing: " + TRUSTED_BENCHMARK_REPORT
)
old_text_only_outputs = [
    Path(RESULTS_ROOT) / "rag_suite_final_gpt",
    Path(RESULTS_ROOT) / "rag_suite_final",
]
for old_text_only in old_text_only_outputs:
    if old_text_only.exists():
        print("Preserving text-only ablation (not imported):", old_text_only)
print("New isolated multimodal output:", Path(RESULTS_ROOT) / FINAL_OUTPUT_NAME)

In [ ]:
import getpass, os
if not os.environ.get("OPENROUTER_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY") or ""
    except Exception:
        pass
if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OPENROUTER_API_KEY: ")
assert os.environ["OPENROUTER_API_KEY"], "OPENROUTER_API_KEY is empty"
print("API key loaded in this runtime only.")

In [ ]:
# Live RAG execution. Rerun this exact cell after any disconnect/pause.
from pathlib import Path
import sys

output = Path(RESULTS_ROOT) / FINAL_OUTPUT_NAME
command = [
    sys.executable, "-u", "-m", "geomapbench_eval", "rag-suite",
    "--benchmark-root", BENCHMARK_ROOT, "--corpus-root", CORPUS_ROOT,
    "--work-root", "/content/geomaprag_multimodal_work_v220",
    "--output", str(output), "--target-per-leaf", str(TARGET_PER_LEAF),
    "--models", str(repo / "config/evaluation_models_final.json"),
    "--benchmark-report", TRUSTED_BENCHMARK_REPORT,
    "--agent-cache", str(Path(CACHE_ROOT) / "agent_cache_multimodal_v220"),
    "--model", ANSWER_MODEL, "--agent-model", AGENT_MODEL,
    "--agent-reasoning-effort", "minimal",
    "--conditions", "multimodal_rag,agentic_multimodal_rag",
    "--max-cost-usd-per-model", str(MAX_COST_USD_PER_CONDITION),
    "--top-k", "3", "--candidate-k", "40", "--image-candidate-k", "20",
    "--max-reference-images", "1",
    "--max-passage-chars", "1200", "--max-context-chars", "3000",
    "--request-delay-seconds", str(REQUEST_DELAY_SECONDS),
    "--progress-every", str(PROGRESS_EVERY),
    "--timeout-seconds", "240", "--retries", "6",
    "--retry-base-seconds", "5", "--retry-max-seconds", "60",
    "--max-consecutive-errors", "2",
]
if Path(BASE_RESULTS).is_file():
    command += ["--base-results", BASE_RESULTS]
run_live(command, cwd=repo)
print("Canonical multimodal RAG suite:", output)

In [ ]:
# Matched conditions, paired comparison and Seaborn figures.
import json, pandas as pd
from IPython.display import Image, display

summary = json.loads((output / "rag_suite_summary.json").read_text(encoding="utf-8"))
rows = []
for condition, report in summary["reports"].items():
    stats = report["analysis"]["condition_summary"].get(condition, {})
    rows.append({
        "condition": condition,
        "complete": report["run"].get("complete"),
        "completed_total": report["run"].get("completed_total"),
        "target_records": report["run"].get("target_records"),
        **stats,
    })
display(pd.DataFrame(rows))
comparisons = summary.get("comparisons", {})
if comparisons:
    display(pd.DataFrame([{"comparison": key, **value} for key, value in comparisons.items()]))
print("Cohort:", {key: summary["cohort"][key] for key in ("target_per_leaf", "target_record_count", "selected_ids_hash")})
print("Trusted benchmark report reused:", summary["benchmark_report"].get("reused_without_rescan"))
print("Multimodal runtime validation:", summary["multimodal_validation"])
audits = {
    condition: report.get("modality_audit", {})
    for condition, report in summary["reports"].items()
}
display(pd.DataFrame([{"condition": key, **value} for key, value in audits.items()]))
for condition, audit in audits.items():
    if summary["reports"][condition]["run"].get("completed_total", 0):
        assert audit.get("both_modalities_observed"), f"Modality audit failed: {condition}"
for name in summary.get("plots", []):
    display(Image(filename=str(output / "plots" / name)))

Start with `TARGET_PER_LEAF=1`. Confirm the explicit `[rag-validation] PASS` line and the modality-audit table, then increase the same value to 50 in the same output directory. Each larger cohort contains all earlier IDs, so prior calls are reused. Do not rename the output and do not copy files from any earlier `rag_suite_final*` directory; those are text-only ablations.